# Lab Assignment 1: Neural Network for Bank Customer Churn Prediction

**Name:** Parimal Ahire  
**PRN:** 202301040067  
**Course:** Deep Learning Lab  
**GitHub Repository:** https://github.com/ParimalAhire/Lab-Assignment-1-Neural-Network


## Aim
To implement a feedforward neural network in three ways to predict **bank customer churn**:
1. **From Scratch** using NumPy (forward pass, backpropagation, gradient descent)
2. **Using Keras** (high-level TensorFlow/Keras Sequential API)
3. **Using Scikit-learn** (MLPClassifier)

The network takes 10 customer features as input and predicts whether the customer will **leave (1)** or **stay (0)**.


## Algorithm
1. Import all required libraries once
2. Load and preprocess the dataset (encoding, scaling, train/test split)
3. **Part 1:** ANN from scratch — forward pass → loss → backprop → gradient descent
4. **Part 2:** ANN using Keras Sequential API with Adam optimizer
5. **Part 3:** ANN using Scikit-learn MLPClassifier with Adam solver
6. Compare accuracy, confusion matrix and classification report across all three
7. Single combined user input test — all three models predict together


---
## Step 1: Import All Libraries
> All imports are done **once here** and shared across all three parts.


In [ ]:
# Core
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Preprocessing
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, confusion_matrix,
    classification_report, ConfusionMatrixDisplay
)

# Keras
from tensorflow import keras
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.optimizers import Adam
import tensorflow as tf

# Scikit-learn MLP
from sklearn.neural_network import MLPClassifier

# Reproducibility
np.random.seed(42)
tf.random.set_seed(42)

print('All libraries imported successfully!')
print('Keras version:', keras.__version__)


## Step 2: Load and Explore Dataset
> Dataset is loaded and explored **once here** — shared by all three parts.

**Dataset:** Bank Customer Churn (10,000 rows, 14 columns)  
**Task:** Binary Classification — predict if customer will leave (1) or stay (0)


In [ ]:
# Load dataset
df = pd.read_csv('Churn_Modelling.csv')

print('Shape:', df.shape)
print('\nFirst 5 rows:')
df.head()


In [ ]:
# Basic info
print('Column Info:')
print(df.dtypes)
print('\nMissing values:', df.isnull().sum().sum())
print('\nTarget Distribution:')
print(df['Exited'].value_counts())
print(f'\nChurn Rate: {df["Exited"].mean()*100:.1f}%')


In [ ]:
# Visualize class distribution
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Count plot
axes[0].bar(['Stayed (0)', 'Churned (1)'],
            df['Exited'].value_counts().values,
            color=['steelblue', 'tomato'], edgecolor='black')
axes[0].set_title('Customer Churn Distribution')
axes[0].set_ylabel('Count')
for i, v in enumerate(df['Exited'].value_counts().values):
    axes[0].text(i, v + 50, str(v), ha='center', fontweight='bold')

# Churn by Geography
churn_geo = df.groupby('Geography')['Exited'].mean() * 100
axes[1].bar(churn_geo.index, churn_geo.values,
            color=['steelblue', 'tomato', 'seagreen'], edgecolor='black')
axes[1].set_title('Churn Rate by Geography (%)')
axes[1].set_ylabel('Churn Rate (%)')
for i, v in enumerate(churn_geo.values):
    axes[1].text(i, v + 0.3, f'{v:.1f}%', ha='center', fontweight='bold')

plt.tight_layout()
plt.show()


## Step 3: Preprocess the Dataset
> Preprocessing is done **once** — `X_train`, `X_test`, `y_train`, `y_test` are shared by all parts.

Steps:
1. Drop irrelevant columns (`RowNumber`, `CustomerId`, `Surname`)
2. Encode categorical columns (`Gender`, `Geography`)
3. Separate features and target
4. Scale features using `StandardScaler`
5. Split into train (80%) and test (20%)


In [ ]:
# 1. Drop irrelevant columns
df = df.drop(['RowNumber', 'CustomerId', 'Surname'], axis=1)

# 2. Encode categorical columns
le = LabelEncoder()
df['Gender']    = le.fit_transform(df['Gender'])      # Female=0, Male=1
df['Geography'] = le.fit_transform(df['Geography'])   # France=0, Germany=1, Spain=2

print('After encoding:')
print(df.head())
print('\nFeature columns:', list(df.drop('Exited', axis=1).columns))


In [ ]:
# 3. Separate features and target
X = df.drop('Exited', axis=1).values
y = df['Exited'].values

# 4. Feature scaling
scaler = StandardScaler()
X = scaler.fit_transform(X)

# 5. Train/test split (80/20)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f'X_train shape : {X_train.shape}')
print(f'X_test  shape : {X_test.shape}')
print(f'y_train shape : {y_train.shape}')
print(f'y_test  shape : {y_test.shape}')
print(f'\nFeatures      : {X_train.shape[1]}')
print(f'Training rows : {X_train.shape[0]}')
print(f'Test rows     : {X_test.shape[0]}')


---
# Part 1: ANN From Scratch (NumPy Only)
---

## Step 4: Define Activation Functions

In [ ]:
def sigmoid(x):
    return 1 / (1 + np.exp(-np.clip(x, -500, 500)))

def sigmoid_derivative(x):
    """Derivative of sigmoid given its output"""
    return x * (1 - x)

def relu(x):
    return np.maximum(0, x)

def relu_derivative(x):
    return (x > 0).astype(float)


## Step 5: Network Configuration & Weight Initialization

**Architecture:** 10 → 6 → 6 → 1
- Input: 10 features
- Hidden Layer 1: 6 neurons (ReLU)
- Hidden Layer 2: 6 neurons (ReLU)
- Output: 1 neuron (Sigmoid — binary classification)


In [ ]:
input_neurons   = X_train.shape[1]  # 10
hidden1_neurons = 6
hidden2_neurons = 6
output_neurons  = 1
learning_rate   = 0.01
epochs          = 100

# initialization
W1 = np.random.randn(input_neurons,   hidden1_neurons) * np.sqrt(2 / input_neurons)
b1 = np.zeros((1, hidden1_neurons))
W2 = np.random.randn(hidden1_neurons, hidden2_neurons) * np.sqrt(2 / hidden1_neurons)
b2 = np.zeros((1, hidden2_neurons))
W3 = np.random.randn(hidden2_neurons, output_neurons)  * np.sqrt(2 / hidden2_neurons)
b3 = np.zeros((1, output_neurons))

print(f'Architecture  : {input_neurons} -> {hidden1_neurons} -> {hidden2_neurons} -> {output_neurons}')
print(f'Learning Rate : {learning_rate} | Epochs: {epochs}')
print(f'W1 shape: {W1.shape} | W2 shape: {W2.shape} | W3 shape: {W3.shape}')


## Step 6: Training (Forward Pass + Backpropagation + Gradient Descent)
> Weight updates printed every **10 epochs**.


In [ ]:
losses_scratch = []

for epoch in range(epochs):

    # Forward Pass
    z1 = np.dot(X_train, W1) + b1
    a1 = relu(z1)

    z2 = np.dot(a1, W2) + b2
    a2 = relu(z2)

    z3 = np.dot(a2, W3) + b3
    a3 = sigmoid(z3)                        

    # Loss
    m = X_train.shape[0]
    loss = -np.mean(
        y_train.reshape(-1,1) * np.log(a3 + 1e-8) +
        (1 - y_train.reshape(-1,1)) * np.log(1 - a3 + 1e-8)
    )
    losses_scratch.append(loss)

    # Backpropagation
    # Output layer
    d3 = a3 - y_train.reshape(-1, 1)
    dW3 = np.dot(a2.T, d3) / m
    db3 = np.sum(d3, axis=0, keepdims=True) / m

    # Hidden layer 2
    d2 = np.dot(d3, W3.T) * relu_derivative(a2)
    dW2 = np.dot(a1.T, d2) / m
    db2 = np.sum(d2, axis=0, keepdims=True) / m

    # Hidden layer 1
    d1 = np.dot(d2, W2.T) * relu_derivative(a1)
    dW1 = np.dot(X_train.T, d1) / m
    db1 = np.sum(d1, axis=0, keepdims=True) / m

    # Weight Update (Gradient Descent)
    W3 -= learning_rate * dW3
    b3 -= learning_rate * db3
    W2 -= learning_rate * dW2
    b2 -= learning_rate * db2
    W1 -= learning_rate * dW1
    b1 -= learning_rate * db1

    # Print weights every 10 epochs
    if (epoch + 1) % 10 == 0:  
        train_pred = (a3 >= 0.5).astype(int).flatten()
        train_acc  = accuracy_score(y_train, train_pred)
        print(f'Epoch {epoch+1:3d} | Loss: {loss:.4f} | Train Accuracy: {train_acc*100:.2f}%')
        print(f'  Updated W1 (first row): {np.round(W1[0], 4)}')
        print(f'  Updated W2 (first row): {np.round(W2[0], 4)}')
        print(f'  Updated W3 (first row): {np.round(W3.flatten()[:3], 4)} ...')
        print()


## Step 7: Evaluate From Scratch Model

In [ ]:
# Predict on test set
z1_t = np.dot(X_test, W1) + b1;  a1_t = relu(z1_t)
z2_t = np.dot(a1_t, W2) + b2;  a2_t = relu(z2_t)
z3_t = np.dot(a2_t, W3) + b3;  a3_t = sigmoid(z3_t)

pred_scratch     = (a3_t >= 0.5).astype(int).flatten()
prob_scratch     = a3_t.flatten()
acc_scratch      = accuracy_score(y_test, pred_scratch)

print('=== From Scratch Results ===')
print(f'Test Accuracy : {acc_scratch*100:.2f}%')
print('\nClassification Report:')
print(classification_report(y_test, pred_scratch, target_names=['Stayed','Churned']))

# Confusion matrix
cm_scratch = confusion_matrix(y_test, pred_scratch)
disp = ConfusionMatrixDisplay(cm_scratch, display_labels=['Stayed','Churned'])
disp.plot(cmap='Blues')
plt.title('Confusion Matrix — From Scratch')
plt.show()


## Step 8: Loss Curve (From Scratch)

In [ ]:
plt.figure(figsize=(8, 4))
plt.plot(losses_scratch, color='blue', linewidth=2)
plt.title('Training Loss Over Epochs (From Scratch)')
plt.xlabel('Epoch')
plt.ylabel('Binary Cross-Entropy Loss')
plt.grid(True)
plt.tight_layout()
plt.show()
print(f'Final Training Loss: {losses_scratch[-1]:.4f}')


---
# Part 2: ANN Using Keras
---

## Step 9: Build & Train Keras Model

- **Architecture:** 10 → 6 (ReLU) → 6 (ReLU) → 1 (Sigmoid)
- **Optimizer:** Adam (adaptive learning rate)
- **Loss:** Binary Cross-Entropy (standard for binary classification)
- **Dropout:** Added to reduce overfitting


In [ ]:
model_keras = Sequential([
    Dense(6, input_dim=10, activation='relu', name='hidden_1'),
    Dropout(0.2),
    Dense(6, activation='relu', name='hidden_2'),
    Dropout(0.2),
    Dense(1, activation='sigmoid', name='output')
], name='ANN_Churn_Keras')

model_keras.compile(
    optimizer=Adam(learning_rate=0.001),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

model_keras.summary()


In [ ]:
history_keras = model_keras.fit(
    X_train, y_train,
    epochs=100,
    batch_size=32,
    validation_data=(X_test, y_test),
    verbose=0
)

print('Training complete!')
print(f'Final Train Loss     : {history_keras.history["loss"][-1]:.4f}')
print(f'Final Val Loss       : {history_keras.history["val_loss"][-1]:.4f}')
print(f'Final Train Accuracy : {history_keras.history["accuracy"][-1]*100:.2f}%')
print(f'Final Val Accuracy   : {history_keras.history["val_accuracy"][-1]*100:.2f}%')


## Step 10: Evaluate Keras Model

In [ ]:
prob_keras  = model_keras.predict(X_test, verbose=0).flatten()
pred_keras  = (prob_keras >= 0.5).astype(int)
acc_keras   = accuracy_score(y_test, pred_keras)

print('=== Keras Results ===')
print(f'Test Accuracy : {acc_keras*100:.2f}%')
print('\nClassification Report:')
print(classification_report(y_test, pred_keras, target_names=['Stayed','Churned']))

# Confusion matrix
cm_keras = confusion_matrix(y_test, pred_keras)
disp = ConfusionMatrixDisplay(cm_keras, display_labels=['Stayed','Churned'])
disp.plot(cmap='Greens')
plt.title('Confusion Matrix — Keras')
plt.show()


## Step 11: Loss & Accuracy Curves (Keras)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Loss
axes[0].plot(history_keras.history['loss'],     color='green', linewidth=2, label='Train')
axes[0].plot(history_keras.history['val_loss'], color='orange',linewidth=2, label='Validation')
axes[0].set_title('Loss Over Epochs (Keras)')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Binary Cross-Entropy Loss')
axes[0].legend()
axes[0].grid(True)

# Accuracy
axes[1].plot(history_keras.history['accuracy'],     color='green', linewidth=2, label='Train')
axes[1].plot(history_keras.history['val_accuracy'], color='orange',linewidth=2, label='Validation')
axes[1].set_title('Accuracy Over Epochs (Keras)')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy')
axes[1].legend()
axes[1].grid(True)

plt.tight_layout()
plt.show()


---
# Part 3: ANN Using Scikit-learn (MLPClassifier)
---

## Step 12: Build & Train MLPClassifier

- **Architecture:** 10 → 6 → 6 → 1
- **Solver:** Adam
- **Activation:** ReLU for hidden layers


In [ ]:
model_sklearn = MLPClassifier(
    hidden_layer_sizes=(6, 6),   
    activation='relu',
    solver='adam',
    learning_rate_init=0.001,
    max_iter=500,
    random_state=42,
    verbose=False
)

model_sklearn.fit(X_train, y_train)
print('Training complete!')
print(f'Iterations ran: {model_sklearn.n_iter_}')


## Step 13: Evaluate Sklearn Model

In [ ]:
prob_sklearn = model_sklearn.predict_proba(X_test)[:, 1]
pred_sklearn = model_sklearn.predict(X_test)
acc_sklearn  = accuracy_score(y_test, pred_sklearn)

print('=== Sklearn Results ===')
print(f'Test Accuracy : {acc_sklearn*100:.2f}%')
print('\nClassification Report:')
print(classification_report(y_test, pred_sklearn, target_names=['Stayed','Churned']))

# Confusion matrix
cm_sklearn = confusion_matrix(y_test, pred_sklearn)
disp = ConfusionMatrixDisplay(cm_sklearn, display_labels=['Stayed','Churned'])
disp.plot(cmap='Oranges')
plt.title('Confusion Matrix — Sklearn')
plt.show()

# Loss curve
plt.figure(figsize=(8, 4))
plt.plot(model_sklearn.loss_curve_, color='darkorange', linewidth=2)
plt.title('Training Loss Over Iterations (Sklearn)')
plt.xlabel('Iteration')
plt.ylabel('Loss')
plt.grid(True)
plt.tight_layout()
plt.show()


---
# Part 4: Comparison of All Three Approaches
---

## Step 14: Accuracy & Confusion Matrix Comparison

In [ ]:
print('=' * 45)
print(f"{'Model':<15} {'Accuracy':^10} {'Churn Recall':^15}")
print('=' * 45)

for name, pred in [('From Scratch', pred_scratch),
                    ('Keras',        pred_keras),
                    ('Sklearn',      pred_sklearn)]:
    acc = accuracy_score(y_test, pred) * 100
    cm  = confusion_matrix(y_test, pred)
    churn_recall = cm[1,1] / (cm[1,0] + cm[1,1]) * 100
    print(f'{name:<15} {acc:^10.2f}% {churn_recall:^15.2f}%')

print('=' * 45)
print('\nChurn Recall = how well the model catches actual churners')


## Step 15: Side-by-Side Confusion Matrices

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

for ax, cm, title, cmap in zip(
    axes,
    [cm_scratch, cm_keras, cm_sklearn],
    ['From Scratch', 'Keras', 'Sklearn'],
    ['Blues', 'Greens', 'Oranges']
):
    disp = ConfusionMatrixDisplay(cm, display_labels=['Stayed', 'Churned'])
    disp.plot(ax=ax, cmap=cmap, colorbar=False)
    ax.set_title(f'Confusion Matrix\n{title}')

plt.tight_layout()
plt.show()


## Step 16: Combined Loss Curves

In [ ]:
plt.figure(figsize=(10, 4))
plt.plot(losses_scratch,                   color='blue',       linewidth=2, label='From Scratch')
plt.plot(history_keras.history['loss'],    color='green',      linewidth=2, label='Keras (Train)')
plt.plot(history_keras.history['val_loss'],color='lightgreen', linewidth=2, label='Keras (Val)', linestyle='--')
plt.plot(model_sklearn.loss_curve_,        color='darkorange', linewidth=2, label='Sklearn')
plt.title('Training Loss Comparison: All Three Models')
plt.xlabel('Epoch / Iteration')
plt.ylabel('Loss')
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()


---
## Step 17: Test All Three Models — Single Customer Input
> Enter a customer's details **once** — all three models predict together.
---

**Input fields:**
| Feature | Range / Values |
|---|---|
| CreditScore | 300 – 850 |
| Geography | 0=France, 1=Germany, 2=Spain |
| Gender | 0=Female, 1=Male |
| Age | 18 – 92 |
| Tenure | 0 – 10 (years with bank) |
| Balance | 0 – 250000 |
| NumOfProducts | 1 – 4 |
| HasCrCard | 0=No, 1=Yes |
| IsActiveMember | 0=No, 1=Yes |
| EstimatedSalary | 0 – 200000 |


In [ ]:
print('Enter customer details:')
credit_score     = float(input('CreditScore      (e.g. 600)     : '))
geography        = float(input('Geography        (0/1/2)         : '))
gender           = float(input('Gender           (0=F, 1=M)      : '))
age              = float(input('Age              (e.g. 40)       : '))
tenure           = float(input('Tenure           (e.g. 3)        : '))
balance          = float(input('Balance          (e.g. 60000)    : '))
num_products     = float(input('NumOfProducts    (1-4)           : '))
has_cr_card      = float(input('HasCrCard        (0/1)           : '))
is_active_member = float(input('IsActiveMember   (0/1)           : '))
salary           = float(input('EstimatedSalary  (e.g. 50000)   : '))

# Build and scale input
customer = np.array([[credit_score, geography, gender, age, tenure,
                       balance, num_products, has_cr_card,
                       is_active_member, salary]])
customer_scaled = scaler.transform(customer)

# From Scratch
z1_c = np.dot(customer_scaled, W1) + b1; a1_c = relu(z1_c)
z2_c = np.dot(a1_c, W2) + b2;            a2_c = relu(z2_c)
z3_c = np.dot(a2_c, W3) + b3;            prob_s = sigmoid(z3_c)[0][0]

# Keras
prob_k = model_keras.predict(customer_scaled, verbose=0)[0][0]

# Sklearn
prob_sk = model_sklearn.predict_proba(customer_scaled)[0][1]

# Display
print('\n' + '='*55)
print(f"{'Customer Churn Prediction':^55}")
print('='*55)
print(f"{'Model':<16} {'Churn Prob':^14} {'Prediction':^14} {'Result':^10}")
print('-'*55)

for model_name, prob in [('From Scratch', prob_s),
                          ('Keras',        prob_k),
                          ('Sklearn',      prob_sk)]:
    label  = 1 if prob >= 0.5 else 0
    result = '⚠ CHURN' if label == 1 else '✓ STAY'
    print(f"{model_name:<16} {prob:^14.4f} {label:^14} {result:^10}")

print('='*55)


## Summary Comparison

| Feature | From Scratch | Keras | Scikit-learn |
|---|---|---|---|
| Library | NumPy | TensorFlow/Keras | scikit-learn |
| Optimizer | Gradient Descent | Adam | Adam |
| Hidden Activation | ReLU | ReLU | ReLU |
| Output Activation | Sigmoid | Sigmoid | Sigmoid |
| Loss Function | Binary Cross-Entropy | Binary Cross-Entropy | Log Loss |
| Regularization | None | Dropout (0.2) | None |
| Backprop | Manual (coded) | Automatic | Automatic |
| Best for | Learning concepts | Production DL | Quick ML baselines |


## Applications

- **Customer churn prediction** — Telecom, banking, SaaS companies
- **Credit risk assessment** — Predict loan defaults
- **Fraud detection** — Flag suspicious transactions
- **Medical diagnosis** — Predict disease likelihood from patient features
- **Recommendation systems** — Netflix, YouTube, Amazon
- **Image recognition** — Facial recognition, object detection


## Conclusion

This assignment implemented a feedforward neural network in three ways to predict bank customer churn:

1. **From Scratch (NumPy):** Manually coded a 3-layer network (10→6→6→1) with ReLU hidden activations and sigmoid output. Forward pass, backpropagation, and gradient descent were all implemented by hand, giving full transparency into weight updates at each epoch.

2. **Keras:** A Sequential model with Dropout regularization was built using just a few lines. Adam optimizer and validation curves helped monitor overfitting. Achieved the best accuracy.

3. **Scikit-learn:** MLPClassifier provided the simplest implementation with direct fit/predict API. Suitable as a quick baseline.

All three models successfully learned to predict customer churn with ~85–86% accuracy. The Keras model benefited from Dropout and validation monitoring, while the From Scratch model demonstrated the core mechanics of neural network training. The single combined prediction cell at Step 17 allows easy real-time testing across all three models.
